# SBERT SentEval Evaluation — Table 5

Evaluate **SBERT-NLI-base** (`bert-base-uncased`, MEAN pooling) on the seven SentEval transfer tasks reported in Table 5 of the Sentence-BERT paper. The paper uses logistic regression with 10-fold cross-validation over the sentence embeddings. fileciteturn11file0

## Paper target

| Task | Paper SBERT-NLI-base |
|---|---:|
| MR | 83.64 |
| CR | 89.43 |
| SUBJ | 94.39 |
| MPQA | 89.86 |
| SST | 88.96 |
| TREC | 89.60 |
| SICK-E | 76.00 |
| **Average** | **87.41** |

These are the Table 5 reference values. fileciteturn11file0

In [1]:
import random
from pathlib import Path
import numpy as np
import pandas as pd
import torch
from transformers import AutoTokenizer, AutoModel
import senteval

print('SentEval imported successfully')

print('PyTorch:', torch.__version__)
print('CUDA:', torch.cuda.is_available())


f:\Programs\PYTHON_310\lib\site-packages\requests\__init__.py:113: RequestsDependencyWarning: urllib3 (2.6.3) or chardet (7.4.3)/charset_normalizer (3.4.4) doesn't match a supported version!
  warnings.warn(
f:\Programs\PYTHON_310\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


SentEval imported successfully
PyTorch: 2.11.0+cu128
CUDA: True


In [2]:
MODEL_NAME = 'bert-base-uncased'
NLI_MODEL_PATH = Path('outputs') / 'sbert_nli_base' / 'model' / 'sbert_nli_base.pt'
TOKENIZER_PATH = Path('outputs') / 'sbert_nli_base' / 'tokenizer'
MAX_LENGTH = 128
BATCH_SIZE = 32
SEED = 42
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

print('Checkpoint:', NLI_MODEL_PATH)
print('Device:', DEVICE)


Checkpoint: outputs\sbert_nli_base\model\sbert_nli_base.pt
Device: cuda


In [3]:
def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

set_seed(SEED)


In [4]:
tokenizer = AutoTokenizer.from_pretrained(TOKENIZER_PATH)
encoder = AutoModel.from_pretrained(MODEL_NAME)

checkpoint = torch.load(NLI_MODEL_PATH, map_location='cpu', weights_only=False)
state = checkpoint['model_state_dict']
bert_state = {k[len('bert.'):]: v for k, v in state.items() if k.startswith('bert.')}
missing, unexpected = encoder.load_state_dict(bert_state, strict=False)

print('Missing:', len(missing))
print('Unexpected:', len(unexpected))
print('Hidden size:', encoder.config.hidden_size)

encoder.to(DEVICE)
encoder.eval()


W0924 09:13:05.150000 15160 site-packages\torch\distributed\elastic\multiprocessing\redirects.py:29] NOTE: Redirects are currently not supported in Windows or MacOs.


Missing: 0
Unexpected: 0
Hidden size: 768


BertModel(
  (embeddings): BertEmbeddings(
    (word_embeddings): Embedding(30522, 768, padding_idx=0)
    (position_embeddings): Embedding(512, 768)
    (token_type_embeddings): Embedding(2, 768)
    (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
    (dropout): Dropout(p=0.1, inplace=False)
  )
  (encoder): BertEncoder(
    (layer): ModuleList(
      (0-11): 12 x BertLayer(
        (attention): BertAttention(
          (self): BertSdpaSelfAttention(
            (query): Linear(in_features=768, out_features=768, bias=True)
            (key): Linear(in_features=768, out_features=768, bias=True)
            (value): Linear(in_features=768, out_features=768, bias=True)
            (dropout): Dropout(p=0.1, inplace=False)
          )
          (output): BertSelfOutput(
            (dense): Linear(in_features=768, out_features=768, bias=True)
            (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
            (dropout): Dropout(p=0.1, inplace=False

## Mean pooling

The paper's default SBERT pooling strategy is MEAN. We use attention-mask-aware mean pooling.

In [5]:
@torch.no_grad()
def encode_sentences(sentences, batch_size=BATCH_SIZE):
    embeddings = []
    for start in range(0, len(sentences), batch_size):
        batch = sentences[start:start + batch_size]
        inputs = tokenizer(batch, padding=True, truncation=True,
                           max_length=MAX_LENGTH, return_tensors='pt')
        inputs = {k: v.to(DEVICE) for k, v in inputs.items()}
        outputs = encoder(**inputs)
        hidden = outputs.last_hidden_state
        mask = inputs['attention_mask'].unsqueeze(-1).float()
        pooled = (hidden * mask).sum(dim=1) / mask.sum(dim=1).clamp(min=1e-9)
        embeddings.append(pooled.cpu())
    return torch.cat(embeddings, dim=0)


In [6]:
sentences = ['A man is eating food.', 'A person is having a meal.']
emb = encode_sentences(sentences)
print('Embedding shape:', emb.shape)
print('Cosine:', torch.nn.functional.cosine_similarity(emb[0:1], emb[1:2]).item())


Embedding shape: torch.Size([2, 768])
Cosine: 0.8563448190689087


## Install SentEval if necessary

SentEval is the official evaluation toolkit referenced by the paper.

## SentEval batcher

SentEval supplies tokenized sentences. We join the tokens, run SBERT, and return NumPy embeddings.

In [7]:
def prepare(params, samples):
    return

def batcher(params, batch):
    sentences = [' '.join(sentence) for sentence in batch]
    return encode_sentences(sentences).numpy()


## Run Table 5 tasks

SentEval performs the downstream classifier training and cross-validation. The task names correspond to the datasets used for the paper's MR, CR, SUBJ, MPQA, SST, TREC and SICK-E results.

In [8]:
import inspect

if not hasattr(inspect, '_original_getargspec'):
    inspect._original_getargspec = inspect.getargspec

    def getargspec_compat(func):
        sig = inspect.signature(func)

        args = []
        varargs = None
        keywords = None
        defaults = []

        for name, param in sig.parameters.items():
            if param.kind == inspect.Parameter.POSITIONAL_ONLY:
                args.append(name)

            elif param.kind == inspect.Parameter.POSITIONAL_OR_KEYWORD:
                args.append(name)
                if param.default is not inspect.Parameter.empty:
                    defaults.append(param.default)

            elif param.kind == inspect.Parameter.VAR_POSITIONAL:
                varargs = name

            elif param.kind == inspect.Parameter.VAR_KEYWORD:
                keywords = name

        defaults = tuple(defaults) if defaults else None

        return inspect.ArgSpec(
            args=args,
            varargs=varargs,
            keywords=keywords,
            defaults=defaults
        )

    inspect.getargspec = getargspec_compat

print("SentEval inspect compatibility patch applied.")

SentEval inspect compatibility patch applied.


In [9]:
params = {
    'task_path': str(Path('..') / 'data' / 'SentEval' / 'data'),
    'usepytorch': True,
    'kfold': 10,
    'classifier': {
        'nhid': 0,
        'optim': 'adam',
        'batch_size': 64,
        'tenacity': 5,
        'epoch_size': 4
    }
}

tasks = [
    'MR',
    'CR',
    'SUBJ',
    'MPQA',
    'SST2',
    'TREC',
    'MRPC'
]

se = senteval.engine.SE(
    params,
    batcher,
    prepare
)

results = {}

for task in tasks:
    print(f'\nRunning: {task}')
    results[task] = se.eval(task)
    print(results[task])


Running: MR
{'devacc': 82.98, 'acc': 82.14, 'ndev': 10662, 'ntest': 10662}

Running: CR
{'devacc': 89.91, 'acc': 88.77, 'ndev': 3775, 'ntest': 3775}

Running: SUBJ
{'devacc': 93.7, 'acc': 93.09, 'ndev': 10000, 'ntest': 10000}

Running: MPQA
{'devacc': 90.33, 'acc': 89.91, 'ndev': 10606, 'ntest': 10606}

Running: SST2
{'devacc': 88.07, 'acc': 88.41, 'ndev': 872, 'ntest': 1821}

Running: TREC
{'devacc': 81.68, 'acc': 88.0, 'ndev': 5452, 'ntest': 500}

Running: MRPC
{'devacc': 77.7, 'acc': 76.23, 'f1': 83.35, 'ndev': 4076, 'ntest': 1725}


In [10]:
rows = []

for task, result in results.items():
    rows.append({
        'dataset': task,
        'our_score': float(result['acc'])
    })

results_df = pd.DataFrame(rows)

results_df

,dataset,our_score
0,MR,82.14
1,CR,88.77
2,SUBJ,93.09
3,MPQA,89.91
4,SST2,88.41
5,TREC,88.00
6,MRPC,76.23


In [11]:
paper = {
    'MR': 83.64,
    'CR': 89.43,
    'SUBJ': 94.39,
    'MPQA': 89.86,
    'SST2': 88.96,
    'TREC': 89.60,
    'MRPC': 76.00
}

comparison = results_df.copy()
comparison['paper_score'] = comparison['dataset'].map(paper)
comparison['difference'] = comparison['our_score'] - comparison['paper_score']
comparison


,dataset,our_score,paper_score,difference
0,MR,82.14,83.64,-1.50
1,CR,88.77,89.43,-0.66
2,SUBJ,93.09,94.39,-1.30
3,MPQA,89.91,89.86,0.05
4,SST2,88.41,88.96,-0.55
5,TREC,88.00,89.60,-1.60
6,MRPC,76.23,76.00,0.23


In [12]:
print('Our average:  ', comparison['our_score'].mean())
print('Paper average:', comparison['paper_score'].mean())
print('Difference:   ', comparison['our_score'].mean() - comparison['paper_score'].mean())


Our average:   86.64999999999999
Paper average: 87.41142857142857
Difference:    -0.7614285714285813


In [13]:
OUTPUT_DIR = Path('outputs') / 'sbert_senteval'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
results_df.to_csv(OUTPUT_DIR / 'senteval_results.csv', index=False)
comparison.to_csv(OUTPUT_DIR / 'senteval_comparison.csv', index=False)
print('Saved:', OUTPUT_DIR.resolve())


Saved: F:\SEM-7\NLP\Project\notebooks\outputs\sbert_senteval


## Reproduction notes

- Model is **SBERT-NLI-base**, not the STS-B-fine-tuned model.
- Pooling is MEAN.
- SentEval performs the downstream classification evaluation.
- The paper reports 10-fold cross-validation for Table 5. fileciteturn11file0
- Differences from the historical numbers can result from the reproduced NLI checkpoint, dataset/toolkit versions, preprocessing, and environment.

**Next:** Table 6 ablation study: MEAN/MAX/CLS pooling and the NLI concatenation configurations. The paper evaluates these configurations with 10 random seeds. fileciteturn12file0